# Epic 7 -- Chromasense Multi-Spectral Integration Overview

This notebook introduces the Chromasense multi-spectral imaging approach used in
Epic 7 for **semiconductor material defect detection**.  Standard single-band
imaging cannot distinguish between material composition defects (delamination,
contamination, oxidation) that look similar in visible light.  Multi-spectral
imaging solves this by capturing reflectance at multiple wavelengths.

## What you will learn

1. Why multi-spectral imaging is needed for semiconductor defect detection
2. The Chromasense wavelength configuration
3. How to generate synthetic multi-spectral images
4. Quick demo: rendering and visualisation

## Prerequisites

- Python 3.12, NumPy, SciPy, OpenCV, matplotlib
- Install: `uv pip install -e ".[epic7]"`

---
## 1. Why Multi-Spectral?

| Defect Type | Visible Light | Multi-Spectral |
|-------------|--------------|----------------|
| Delamination | Hard to see (subtle contrast) | Clear spectral shift in blue/UV |
| Contamination | May look like shadow | Distinct contaminant spectrum |
| Oxidation | Slight discolouration | Strong IR reflectance drop |

Chromasense illuminates at 4 wavelengths: **450 nm** (blue), **550 nm** (green),
**650 nm** (red), and **850 nm** (near-IR), capturing a spectral fingerprint per pixel.

In [ ]:
from udm_epic7.spectral.wavelength_model import default_spectral_config

cfg = default_spectral_config()
print(f"Wavelengths: {cfg.wavelengths} nm")
print(f"Materials:   {cfg.material_names}")
print(f"Channels:    {cfg.n_channels}")

---
## 2. Quick Demo: Generate and Visualise

Let's generate a single multi-spectral image and convert it to RGB for viewing.

In [ ]:
from udm_epic7.data.dataset import SpectralDataset
from udm_epic7.rendering.spectral_renderer import spectral_to_rgb
import matplotlib.pyplot as plt

ds = SpectralDataset(n_samples=5, height=256, width=256, seed=0)
sample = ds[0]

print(f"Image shape: {sample['image'].shape}")
print(f"Mask shape:  {sample['mask'].shape}")
print(f"Defect type: {sample['defect_type']}")

rgb = spectral_to_rgb(sample['image'].numpy(), cfg)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rgb)
axes[0].set_title('RGB Visualisation')
axes[1].imshow(sample['image'][0].numpy(), cmap='viridis')
axes[1].set_title(f'{cfg.wavelengths[0]} nm')
axes[2].imshow(sample['mask'][0].numpy(), cmap='gray')
axes[2].set_title('Defect Mask')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## 3. Project Structure

| Module | Description |
|--------|------------|
| `udm_epic7.spectral.wavelength_model` | `SpectralConfig`, reflectance look-up |
| `udm_epic7.spectral.defect_spectra` | Defect-modified spectra |
| `udm_epic7.rendering.spectral_renderer` | Multi-channel image rendering |
| `udm_epic7.data.dataset` | `SpectralDataset`, dataset generation |
| `udm_epic7.evaluation.spectral_metrics` | SAM, anomaly scoring |
| `udm_epic7.cli_epic7` | CLI: generate, preview, evaluate, visualize-spectrum |

## Next notebooks

| Notebook | Topic |
|----------|------|
| `epic7_01_spectral_model.ipynb` | Material spectra deep dive |
| `epic7_02_defect_spectra.ipynb` | Defect spectral signatures |
| `epic7_03_rendering.ipynb` | Multi-spectral rendering pipeline |